# Making unichart plots look like Matplotlib

unichart draws with **Plotly**, which has a recognizable house style: no axis
spines, a drawn zero line, its own color cycle, its own font. When your figures
have to sit next to Matplotlib output &mdash; a paper, a report, a deck already
full of `pyplot` &mdash; one call re-themes the whole environment:

```python
nb.set_plot_style('matplotlib')   # or 'mpl' / 'plt'
nb.set_plot_style('plotly')       # back to the default look
```

This notebook shows what changes, what deliberately doesn't, and how the style
interacts with the rest of the formatting API.

## 0. Setup &mdash; three runs of the same rig

Nothing style-specific here: one DataFrame, one dataset per run, the usual
`load_df`.

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
from unichart import UnichartNotebook

rng = np.random.default_rng(7)
frames = []
for i, label in enumerate(['baseline', 'coated', 'coated + shroud']):
    t = np.linspace(0, 30, 60)
    rpm = 4800 + 900 * np.sin(t / 4) + 70 * i + rng.normal(0, 25, t.size)
    load = rng.uniform(20, 100, t.size)                 # scattered, for the contour
    frames.append(pd.DataFrame({
        'SETNUMBER': i,
        'TITLE':     label,
        'TIME':      t,
        'CHT':       300 + 45 * (1 - np.exp(-t / 8)) * (1 + 0.10 * i) + rng.normal(0, 1.2, t.size),
        'RPM':       rpm,
        'FUEL':      11.5 - 0.4 * i + 0.9 * np.sin(t / 5) + rng.normal(0, 0.12, t.size),
        'LOAD':      load,
        # A smooth field over (RPM, LOAD) so the contour section has something
        # map-shaped to draw; the time series above stay clean.
        'EFF':       86 + 1.5 * i - 4e-6 * (rpm - 5200) ** 2 - 0.0016 * (load - 75) ** 2,
    }))

nb = UnichartNotebook()
nb.load_df(pd.concat(frames, ignore_index=True))
nb.set_default_format(figsize=(11, 4.5))   # wide and short reads well inline
nb.list_sets()

UniChart Notebook Environment Initialized.
Loaded Set 0: baseline
Loaded Set 1: coated
Loaded Set 2: coated + shroud

Loaded Datasets:
Set    Title            Selected  Shape   Query?  
--------------------------------------------------
Set 0  baseline         ✓         60 x 8  None    
Set 1  coated           ✓         60 x 8  None    
Set 2  coated + shroud  ✓         60 x 8  None    


## 1. The default look

Plotly's: unframed plot area, faint grid, a drawn zero line where one falls in
range, Plotly's qualitative palette, Open Sans.

In [2]:
nb.plot(x='TIME', y=['CHT', 'RPM'], suptitle='Rig sweep — default (Plotly) style')

## 2. One call

Same plot call, different look: white plot area framed by spines on all four
sides, ticks pointing outward, **no** zero line, a gray grid, DejaVu Sans at
Matplotlib's point sizes, and the **tab10** color cycle.

In [3]:
nb.set_plot_style('matplotlib')
nb.plot(x='TIME', y=['CHT', 'RPM'], suptitle='Rig sweep — matplotlib style')

Plot style set to: matplotlib (existing datasets restyled)


## 3. What actually changed

A plot style has to work through **two** mechanisms, because a Plotly template
can only carry *layout* defaults:

| Half | Reaches | Applied |
|---|---|---|
| **Layout** &mdash; a Plotly template | backgrounds, spines, ticks, grid, fonts | to every figure as it is finalized |
| **Per-trace** &mdash; `color_map` + `default_format` | dataset colors, markersize, hue colorscale | to the datasets themselves |

The second half is why `set_plot_style` touches your loaded datasets: colors and
marker sizes are written explicitly into each trace from the `Dataset`, so no
template could reach them.

In [4]:
print('plot_style :', nb.plot_style)
print('color_map  :', list(nb.color_map[:5]), '...')
print('set colors :', [ds.color for ds in nb.sets])
print('markersize :', [ds.markersize for ds in nb.sets])   # 6 pt -> 8.3 px
print('hue_palette:', [ds.hue_palette for ds in nb.sets])  # Jet -> Viridis

plot_style : matplotlib
color_map  : ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'] ...
set colors : ['#1f77b4', '#ff7f0e', '#2ca02c']
markersize : [8.3, 8.3, 8.3]
hue_palette: ['Viridis', 'Viridis', 'Viridis']


## 4. Dark mode stays orthogonal

The style is **not** a replacement for `toggle_darkmode` &mdash; each style has a
light and a dark variant, so the two compose. Dark matplotlib is
`dark_background`'s black figure with white spines and text.

In [5]:
nb.toggle_darkmode(True)
nb.plot(x='TIME', y='CHT', suptitle='matplotlib style, dark mode')

Plot theme set to: Dark Mode


In [6]:
nb.toggle_darkmode(False)   # back to light for the rest of the notebook

Plot theme set to: Light Mode


## 5. Colors unichart picks for you follow the style too

Anywhere unichart chooses a color or a colorscale on your behalf, it goes
through the style:

- **Contours and hue-colored scatters** use `hue_palette`, which the Matplotlib
  style moves from Jet to **viridis** (`rcParams['image.cmap']`). Override it per
  dataset with `nb.hue_palette('all', 'Jet')` if you want the old ramp back.
- **`style_by=`** on `plot_ymult` auto-differentiates the y variables by cycling
  `color_map` / `marker_map` &mdash; so it cycles tab10 under this style.

In [7]:
nb.contour(x='RPM', y='LOAD', z='EFF', suptitle='Efficiency map — viridis, not Jet')

In [8]:
nb.plot_ymult(x='TIME', y=['CHT', 'FUEL'], style_by='color+marker',
              suptitle='style_by cycles the active palette')

## 6. Your own formatting still wins

The style only supplies *defaults*. Everything you set explicitly &mdash;
`color`, `linestyle`, `markersize`, `set_font_sizes`, decorations &mdash; sits on
top of it, exactly as it does under the default style.

In [9]:
nb.color(0, 'crimson')                  # one set deliberately off-palette
nb.linestyle(1, '--')                   # dashed line for the coated run
nb.set_font_sizes(suptitle=22)          # bigger than the style's 16.7 px
nb.line('CHT', level=330, color='gray', linestyle=':', label='limit')

nb.plot(x='TIME', y=['CHT', 'RPM'], suptitle='Style + your overrides')

Note that the style's font sizes are *not* written into your settings:
`get_font_sizes` keeps reporting only what **you** set, and
`set_font_sizes(reset=True)` clears only that &mdash; the style's defaults survive
the reset.

In [10]:
print('after set_font_sizes(suptitle=22):', nb.get_font_sizes())
nb.set_font_sizes(reset=True)
print('after reset            :', nb.get_font_sizes())

after set_font_sizes(suptitle=22): {'suptitle': 22.0, 'footer': None, 'legend': None, 'axes_title': None, 'axes_tick': None, 'subplot_title': None, 'colorbar': None, 'hover': None, 'table_header': None, 'table_cell': None}
after reset            : {'suptitle': None, 'footer': None, 'legend': None, 'axes_title': None, 'axes_tick': None, 'subplot_title': None, 'colorbar': None, 'hover': None, 'table_header': None, 'table_cell': None}


## 7. `sets=False` &mdash; restyle the layout, keep the datasets

By default `set_plot_style` re-derives the loaded datasets' color, markersize
and hue palette, which **clears manual overrides** on them (that `crimson` from
section 6 included). When you've hand-picked colors you want to keep, pass
`sets=False`: only the layout and *future* loads follow the new style.

In [11]:
nb.reset_format('sets', 'lines')            # drop the overrides from section 6
nb.color(0, 'black')                        # a deliberate, hand-picked color

nb.set_plot_style('plotly', sets=False)     # Plotly frame, matplotlib colors kept
nb.plot(x='TIME', y='CHT', suptitle='sets=False — layout reverted, datasets untouched')

Reset: dataset formatting, lines.
Plot style set to: plotly


In [12]:
print('style      :', nb.plot_style)
print('set colors :', [ds.color for ds in nb.sets])   # still tab10 + the manual black

style      : plotly
set colors : ['black', '#ff7f0e', '#2ca02c']


## 8. Going back

`set_plot_style('plotly')` (aliases `'default'`, `'reset'`) restores both halves.
The style is a *default*, so it also comes along with the bulk resets:
`reset_format('defaults')` and `reset_format('all')` return to `'plotly'`.

In [13]:
nb.set_plot_style('plotly')
nb.plot(x='TIME', y=['CHT', 'RPM'], suptitle='Back to the default look')

Plot style set to: plotly (existing datasets restyled)


## 9. Notes

- **Markers stay on.** Matplotlib draws line plots without markers; unichart
  assigns one per dataset so runs stay distinguishable. Match Matplotlib with
  `nb.set_default_format(marker=None)`.
- **Grid visibility is yours.** Matplotlib defaults `axes.grid` to `False`, but
  unichart's own `grid=` argument defaults to on; the style only restyles the
  grid (color `#b0b0b0`, solid, 0.8 px) rather than fighting an explicit choice.
- **Dashboards keep the board font.** `nb.dashboard(...)` panels carry the
  spines, palette and backgrounds, but keep the board's UI font so charts and
  chrome read as one surface.
- **Under the hood** the style registers two Plotly templates,
  `unichart_matplotlib` and `unichart_matplotlib_dark`, so
  `fig.update_layout(template='unichart_matplotlib')` also works on any figure
  you build yourself.
- **Point sizes** convert as `px = pt * 100 / 72`, because unichart maps one
  `figsize` inch to 100 px &mdash; Matplotlib's own default DPI. So Matplotlib's
  10 pt tick labels land at 13.9 px.